# 20260915-OpenMP continued

## Parallelizing dot product continued

<div class="alert alert-block alert-info">
<b></b> 
This continues from the previous notes, where we were going over different ways of vectorizing the dot function
</div>

### Reductions

- You know what else is annoying? Creating individual sum areas for each thread
    - Have to be careful about false sharing
    - Needs to be flexible to the number of threads being used


- We can use the `reduction` modifier for `omp parallel for` as well

```c
double dot_opt4(size_t n, const double *a, const double *b) {
  double sum = 0;
  omp_set_num_threads(4);
  #pragma omp parallel for reduction(+:sum)
  for (size_t i=0; i<n; i++)
    sum += a[i] * b[i];
  return sum;
}
```

How does it work?

![reduction_clause_diagram](./reduction_clause.png)

*from http://jakascorner.com/blog/2016/06/omp-for-reduction.html*

- Note that the local `sum` variable for each thread is completely normal
- OpenMP handles the false sharing issue without us needing to do anything


### Performance of our different optimized dot products

| `dot` version | Description |
| --- | --- |
| `ref`  | completely serial implementation |
| `opt1` |  naive parallelization, with interleaving data |
| `opt2` |  padding and manual chunking/scheduling |
| `opt3` |  `omp parallel for` with static scheduling |
| `opt4` |  same as `opt3`, but with reduction clause |

In [6]:
!make CFLAGS='-O3 -march=native -fopenmp' -B dot
!OMP_NUM_THREADS=2 ./dot -r 4 -n 1000000

cc -O3 -march=native -fopenmp    dot.c   -o dot
  Name  	flops	ticks	flops/tick
 dot_ref	2000000	4057726	    0.49	
 dot_ref	2000000	3455875	    0.58	
 dot_ref	2000000	3266452	    0.61	
 dot_ref	2000000	2771885	    0.72	

dot_opt1	2000000	2205205	    0.91	
dot_opt1	2000000	2032270	    0.98	
dot_opt1	2000000	1908097	    1.05	
dot_opt1	2000000	1858887	    1.08	

dot_opt2	2000000	793373	    2.52	
dot_opt2	2000000	827521	    2.42	
dot_opt2	2000000	774900	    2.58	
dot_opt2	2000000	758207	    2.64	

dot_opt3	2000000	789678	    2.53	
dot_opt3	2000000	827487	    2.42	
dot_opt3	2000000	840292	    2.38	
dot_opt3	2000000	885605	    2.26	

dot_opt4	2000000	832850	    2.40	
dot_opt4	2000000	797991	    2.51	
dot_opt4	2000000	808423	    2.47	
dot_opt4	2000000	782563	    2.56	


- We see `opt1` does about twice as better performance than `ref`
    - Makes sense, as we're using two threads
- `opt2` performs much better than `opt1`
    - The data interleaving in `opt1` was causing memory contention issues
    - Both threads wanting to access data on the same cache line
- `opt2`, `opt3`, and `opt4` perform very similarly
    - They were all executing the same strategy
    - The only difference is that the latter optimized runs were using more convienient OpenMP constructs to implement them

## Vectorization Failures - Aliasing

OpenMP-4.0 added the `omp simd` construct, which is a portable way to request that the compiler vectorize code.
An example of a reason why a compiler might fail to vectorize code is aliasing, which we investigate below.

In [20]:
render_c('triad.c')

```c
void triad(int N, double *a, const double *b, double scalar, const double *c) {
    for (int i=0; i<N; i++)
        a[i] = b[i] + scalar * c[i];
}
```


In [30]:
!gcc -O2 -ftree-vectorize -fopt-info-all -c triad.c

Unit growth for small function inlining: 15->15 (0%)

Inlined 0 calls, eliminated 0 functions

BB 3 is always executed in loop 1
loop 1's coldest_outermost_loop is 1, hotter_than_inner_loop is NULL
consider run-time aliasing test between *_3 and *_8
consider run-time aliasing test between *_5 and *_8
triad.c:2:20: optimized: loop vectorized using 16 byte vectors and unroll factor 2
triad.c:2:20: optimized:  loop versioned for vectorization because of possible aliasing
triad.c:1:6: note: vectorized 1 loops in function.
triad.c:2:20: optimized: loop turned into non-loop; it never loops
triad.c:1:6: note: ***** Analysis failed with vector mode V2DF
triad.c:1:6: note: ***** The result for vector mode V16QI would be the same
triad.c:1:6: note: ***** Re-trying analysis with vector mode V8QI
triad.c:1:6: note: ***** Analysis failed with vector mode V8QI
triad.c:1:6: note: ***** Re-trying analysis with vector mode V4QI
triad.c:1:6: note: ***** Analysis failed with vector mode V4QI
BB 8 is alwa

* gcc autovectorization starts at `-O3` or if you use `-ftree-vectorize`
* options such as [-fopt-info](https://gcc.gnu.org/onlinedocs/gcc/Developer-Options.html#index-fopt-info) give useful diagnostics, but are compiler-dependent and sometimes referring to assembly is useful
* `man gcc` with search (`/`) is your friend

### What is aliasing?

Is this valid code?  What xs `x` after this call?
```c
void triad(int N, double *a, const double *b, double scalar, const double *c) {
    for (int i=0; i<N; i++)
        a[i] = b[i] + scalar * c[i];
}

double x[5] = {1, 2, 3, 4, 5};
triad(2, &x[1], x, 10., x);
```

- C allows memory to overlap arbitrarily.
- Here, the `a` parameter is set with the part of data *ahead* of the original array
    - So parts of `b` and `c` are being overwritten before 
- Clearly, that's not what we ever want to happen, but compiler has to respect it

- You can inform the compiler there there shouldn't be any memory overlap using the [`restrict` qualifier](https://en.wikipedia.org/wiki/Restrict)
    - (C99/C11; `__restrict` or `__restrict__` work with most C++ and [CUDA](https://devblogs.nvidia.com/cuda-pro-tip-optimize-pointer-aliasing/) compilers).

In [22]:
render_c('triad-restrict.c')

```c
void triad(int N, double *restrict a, const double *restrict b, double scalar, const double *restrict c) {
    for (int i=0; i<N; i++)
        a[i] = b[i] + scalar * c[i];
}
```


In [23]:
!gcc -O2 -march=native -ftree-vectorize -fopt-info-all -c triad-restrict.c

Unit growth for small function inlining: 15->15 (0%)

Inlined 0 calls, eliminated 0 functions

BB 3 is always executed in loop 1
loop 1's coldest_outermost_loop is 1, hotter_than_inner_loop is NULL
triad-restrict.c:2:20: optimized: loop vectorized using 32 byte vectors and unroll factor 4
triad-restrict.c:2:20: optimized: epilogue loop vectorized using 16 byte vectors and unroll factor 2
triad-restrict.c:1:6: note: vectorized 1 loops in function.
triad-restrict.c:2:20: optimized: loop turned into non-loop; it never loops
triad-restrict.c:3:17: optimized: loop turned into non-loop; it never loops
triad-restrict.c:1:6: note: ***** Analysis failed with vector mode V4DF
triad-restrict.c:1:6: note: ***** The result for vector mode V32QI would be the same
triad-restrict.c:1:6: note: ***** Re-trying analysis with vector mode V16QI
triad-restrict.c:1:6: note: ***** Analysis failed with vector mode V16QI
triad-restrict.c:1:6: note: ***** Re-trying analysis with vector mode V8QI
triad-restrict.c

Notice how there is no more `loop versioned for vectorization because of possible aliasing`.

The complexity of checking for aliasing can grow combinatorially in the number of arrays being processed, leading to many loop variants and/or preventing vectorization.

Can also see this with the multiblock code as well: https://godbolt.org/z/P39984Ko8

Note that this does not significantly affect performance of multiblock. Why might that be?

- Because the aliasing issue occurs well outside the hot loop
- Aliasing is a more significant problem if, for example, the hot loop in the middle was replaced by a function call.
- **The key problem with "aliasing" is that loops may not be vectorized, not necessarily that the loop gets "versioned"**

#### Aside: Warnings
The `-Wrestrict` flag (included in `-Wall`) can catch some programming errors
```c
void foo(double *x) {
  triad(2, x, x, 10, x);
}
```

In [27]:
!gcc -O2 -Wall -c triad-foo.c

triad-foo.c: In function ‘foo’:
triad-foo.c:7:5: warning: passing argument 2 to ‘restrict’-qualified parameter aliases with arguments 3, 5 []8;;https://gcc.gnu.org/onlinedocs/gcc-16.2.0/gcc/Warning-Options.html#index-Wno-restrict-Wrestrict]8;;]
    7 |     triad(2, x, x, 10, x);
      |     ^~~~~


The powers of `-Wrestrict` are limited, however, and (as of gcc-16) do not even catch
```c
void foo(double *x) {
  triad(2, &x[1], x, 10, x);
}
```

### Check the assembly

In [41]:
!gcc -march=native -O2 -ftree-vectorize -c triad.c
!objdump -d --prefix-addresses -M intel triad.o


triad.o:     file format elf64-x86-64


Disassembly of section .text:
0000000000000000 <triad> vmovapd xmm2,xmm0
0000000000000004 <triad+0x4> vbroadcastsd ymm3,xmm0
0000000000000009 <triad+0x9> test   edi,edi
000000000000000b <triad+0xb> jle    00000000000000c3 <triad+0xc3>
0000000000000011 <triad+0x11> cmp    edi,0x1
0000000000000014 <triad+0x14> je     00000000000000d0 <triad+0xd0>
000000000000001a <triad+0x1a> lea    rax,[rsi-0x8]
000000000000001e <triad+0x1e> mov    r8,rax
0000000000000021 <triad+0x21> sub    r8,rdx
0000000000000024 <triad+0x24> cmp    r8,0x10
0000000000000028 <triad+0x28> jbe    00000000000000d0 <triad+0xd0>
000000000000002e <triad+0x2e> sub    rax,rcx
0000000000000031 <triad+0x31> cmp    rax,0x10
0000000000000035 <triad+0x35> jbe    00000000000000d0 <triad+0xd0>
000000000000003b <triad+0x3b> lea    eax,[rdi-0x1]
000000000000003e <triad+0x3e> mov    r9d,edi
0000000000000041 <triad+0x41> cmp    eax,0x2
0000000000000044 <triad+0x44> jbe    00000000000000fd <triad+0

In [40]:
!gcc -march=native -O2 -ftree-vectorize -c triad-restrict.c
!objdump -d --prefix-addresses -M intel triad-restrict.o


triad-restrict.o:     file format elf64-x86-64


Disassembly of section .text:
0000000000000000 <triad> mov    r9d,edi
0000000000000003 <triad+0x3> vmovapd xmm3,xmm0
0000000000000007 <triad+0x7> vbroadcastsd ymm2,xmm0
000000000000000c <triad+0xc> mov    rdi,rdx
000000000000000f <triad+0xf> test   r9d,r9d
0000000000000012 <triad+0x12> jle    000000000000009f <triad+0x9f>
0000000000000018 <triad+0x18> lea    eax,[r9-0x1]
000000000000001c <triad+0x1c> cmp    eax,0x2
000000000000001f <triad+0x1f> jbe    00000000000000a3 <triad+0xa3>
0000000000000025 <triad+0x25> mov    r8d,r9d
0000000000000028 <triad+0x28> vmovapd ymm1,ymm2
000000000000002c <triad+0x2c> xor    eax,eax
000000000000002e <triad+0x2e> shr    r8d,0x2
0000000000000032 <triad+0x32> mov    edx,r8d
0000000000000035 <triad+0x35> shl    rdx,0x5
0000000000000039 <triad+0x39> nop    DWORD PTR [rax+0x0]
0000000000000040 <triad+0x40> vmovupd ymm0,YMMWORD PTR [rcx+rax*1]
0000000000000045 <triad+0x45> vfmadd213pd ymm0,ymm1,YMMWORD PTR [rd

* How do the results change if you go up and replace `-march=native` with `-march=skylake-avx512 -mprefer-vector-width=512`?
* Is the assembly qualitatively different without `restrict` (in which case the compiler "versions" the loop).

### Pragma `omp simd`

An alternative (or supplement) to `restrict` is `#pragma omp simd`.

In [32]:
render_c('triad-omp-simd.c')

```c
void triad(int N, double *a, const double *b, double scalar, const double *c) {
#pragma omp simd
    for (int i=0; i<N; i++)
        a[i] = b[i] + scalar * c[i];
}
```


In [33]:
!gcc -O2 -march=native -ftree-vectorize -fopenmp -fopt-info-all -c triad-omp-simd.c

Unit growth for small function inlining: 15->15 (0%)

Inlined 0 calls, eliminated 0 functions

BB 3 is always executed in loop 1
loop 1's coldest_outermost_loop is 1, hotter_than_inner_loop is NULL
consider run-time aliasing test between *_8 and *_16
consider run-time aliasing test between *_11 and *_16
triad-omp-simd.c:4:17: optimized: loop vectorized using 32 byte vectors and unroll factor 4
triad-omp-simd.c:4:17: optimized: epilogue loop vectorized using 16 byte vectors and unroll factor 2
triad-omp-simd.c:1:6: note: vectorized 1 loops in function.
triad-omp-simd.c:4:17: optimized: loop turned into non-loop; it never loops
triad-omp-simd.c:4:17: optimized: loop turned into non-loop; it never loops
triad-omp-simd.c:1:6: note: ***** Analysis failed with vector mode V4DF
triad-omp-simd.c:1:6: note: ***** The result for vector mode V32QI would be the same
triad-omp-simd.c:1:6: note: ***** Re-trying analysis with vector mode V16QI
triad-omp-simd.c:1:6: note: ***** Analysis failed with ve

In [43]:
!gcc -march=native -O2 -ftree-vectorize -fopenmp-c triad-omp-simd.c
!objdump -d --prefix-addresses -M intel triad-omp-simd.o


triad-omp-simd.o:     file format elf64-x86-64


Disassembly of section .text:
0000000000000000 <triad> mov    r9d,edi
0000000000000003 <triad+0x3> vmovapd xmm3,xmm0
0000000000000007 <triad+0x7> vbroadcastsd ymm2,xmm0
000000000000000c <triad+0xc> mov    rdi,rdx
000000000000000f <triad+0xf> test   r9d,r9d
0000000000000012 <triad+0x12> jle    000000000000009f <triad+0x9f>
0000000000000018 <triad+0x18> lea    eax,[r9-0x1]
000000000000001c <triad+0x1c> cmp    eax,0x2
000000000000001f <triad+0x1f> jbe    00000000000000a8 <triad+0xa8>
0000000000000025 <triad+0x25> mov    r8d,r9d
0000000000000028 <triad+0x28> vmovapd ymm1,ymm2
000000000000002c <triad+0x2c> xor    eax,eax
000000000000002e <triad+0x2e> shr    r8d,0x2
0000000000000032 <triad+0x32> mov    edx,r8d
0000000000000035 <triad+0x35> shl    rdx,0x5
0000000000000039 <triad+0x39> nop    DWORD PTR [rax+0x0]
0000000000000040 <triad+0x40> vmovupd ymm0,YMMWORD PTR [rcx+rax*1]
0000000000000045 <triad+0x45> vfmadd213pd ymm0,ymm1,YMMWORD PTR [rd